In [1]:
%pip install -U langchain langchain-core "langchain-community<=0.4" langchain-anthropic rapidfuzz langchain-experimental langchain-text-splitters langchain-neo4j langchain-huggingface langchain-ollama neo4j python-dotenv sentence-transformers ragas

  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached rapidfuzz-3.14.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_experimental-0.4.1-py3-none-any.whl.metadata (1.3 kB)


In [1]:
from dotenv import load_dotenv
import os
import json

from copy import deepcopy


from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import Node, Relationship

from typing import List, Dict, Any

from pydantic import BaseModel, Field, ValidationError
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.graph_document import Node, Relationship, GraphDocument

from ragas import EvaluationDataset

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

load_dotenv()

True

In [2]:
from langchain_anthropic import ChatAnthropic

cloud_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

In [3]:
graph = Neo4jGraph(database="cloud4")

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

In [6]:
# def serialize_node(node):
#     return Node(id=node["id"], type=node["type"], properties=node["properties"])


# def serialize_relationship(relation, nodes: list[Node]):
#     source = Node(id=relation["source"]["id"], type=relation["source"]["type"], properties=relation["source"]["properties"])
#     target = Node(id=relation["target"]["id"], type=relation["target"]["type"], properties=relation["target"]["properties"])
#     return Relationship(
#         source=source,
#         target=target,
#         type=relation["type"],
#         properties=relation["properties"],
#     )

# def cast_to_graph_document(json_object:any):
#     nodes: list[Node] = []
#     for node in json_object["nodes"]:
#         nodes.append(serialize_node(node))

#     relationships: list[Relationship] = []
#     for rel in json_object["relationships"]:
#         relationships.append(serialize_relationship(rel, nodes))
    
#     source: Document = Document(metadata=json_object["document"]["metadata"], page_content=json_object["document"]["page_content"])
#     return GraphDocument(nodes=nodes, relationships=relationships, source=source)


def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents

In [7]:
# class NodeModel(BaseModel):
#     id: str
#     type: str
#     properties: Dict[str, Any] = Field(default_factory=dict)


# class RelationshipModel(BaseModel):
#     source: NodeModel
#     target: NodeModel
#     type: str
#     properties: Dict[str, Any] = Field(default_factory=dict)


# class DocumentModel(BaseModel):
#     metadata: Dict[str, Any] = Field(default_factory=dict)
#     page_content: str


class ExtractionResult(BaseModel):
    nodes: List[Node] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)
    document: Document

In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a high-recall literary information extraction system for building a knowledge graph from a single source document.

Your task:
Extract all important entities, attributes, and relationships explicitly stated in the input document, and return exactly one valid JSON object with the keys:
- nodes
- relationships
- document

Primary objective:
Maximize useful recall for downstream question answering over a literary corpus.
Prefer extracting too many relevant explicit facts over omitting important ones.

Hard rules:
- Output only valid JSON.
- Do not output markdown, code fences, commentary, or explanations.
- Do not output Python objects such as GraphDocument(...).
- Use only facts explicitly stated in the input document.
- Do not invent facts.
- Do not use background knowledge from outside the document.
- Process the entire input document as one unit.
- If no entities or relationships are present, return empty lists.

Strict schema:
- Every node.type must be exactly one of:
  Person, Organization, Place, Document, Artifact, Event, Case, Role
- Every relationship.type must be exactly one of:
  WORKS_FOR, HAS_ROLE, ALIAS_OF, AFFILIATED_WITH, LOCATED_IN, RESIDES_AT, PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE
- Do not use any other node types or relationship types.

Critical property constraint:
- The properties field of every node and relationship must be a flat JSON object.
- Allowed property values are only:
  - string
  - number
  - boolean
  - list of strings
  - list of numbers
  - list of booleans
- Do NOT use nested objects, nested dictionaries, maps, or lists of objects inside properties.
- If a detail would naturally be structured as an object, flatten it into one or more primitive properties instead.
- Example: use "hair_color": "black" and "face_description": "pale, refined" instead of nested appearance objects.
- Example: use "surface_forms": ["Holmes", "Mr. Holmes"] instead of nested alias objects.

What to extract:
Extract all explicit, question-relevant information, including:

1. Persons
- named characters
- partially named characters if no fuller name appears
- unnamed but clearly important referential persons, e.g. "the landlady", "the old man", "the bride", "the doctor"

2. Places
- addresses, rooms, buildings, streets, cities, regions, countries
- residences and places visited

3. Organizations and groups
- families, houses, institutions, police, royal houses, clubs, employers, professions where relevant

4. Important objects and documents
- letters, notes, photographs, weapons, tools, clothing, disguises, jewelry, keys, furniture, animals, vehicles, etc.
- include plot-relevant objects even if not named formally

5. Events and actions
- meetings, departures, marriages, threats, discoveries, thefts, disguises, conversations, journeys, attacks, investigations, requests, observations
- create event nodes when the event itself is important for later reasoning or connects multiple entities

6. Descriptive attributes
Store important explicit attributes in node.properties, especially:
- physical appearance: age, approximate age, hair, beard, eyes, face, complexion, scars, build, height, clothing, posture, expression
- identity and naming: titles, aliases, alternate names, epithets, descriptions
- social and personal attributes: occupation, profession, rank, family role, marital status, nationality, social status
- emotional or mental state: frightened, agitated, calm, angry, drunk, ill, tired, etc.
- location or residence
- possessions or associated objects
- distinguishing features useful for retrieval or disambiguation
- any other explicit traits likely to matter for question answering

Coreference and canonicalization:
- Resolve pronouns and shortened mentions to the most complete explicit name that appears in the same document.
- Examples:
  - "Holmes", "Mr. Holmes", "he" -> "Sherlock Holmes" if "Sherlock Holmes" appears in the document
  - "Watson", "I", "the narrator" -> "Dr. Watson" if "Dr. Watson" appears in the document
  - "the woman" -> "Irene Adler" only if "Irene Adler" explicitly appears in the same document and the reference is clearly unambiguous
- If the fuller canonical name does not appear in the document, keep the explicit local mention as its own node.
- Do not merge uncertain identities.
- Prefer preserving a distinct explicit entity over making a wrong merge.

Allowed node schemas:
- Person:
  - id: canonical human-readable name
  - type: "Person"
  - properties may include: canonical_name, surface_forms, gender, nationality, description, age, occupation, role, title, marital_status, residence, appearance, hair_color, beard, eye_description, face_description, complexion, build, height_description, scar_description, posture, expression, emotional_state, aliases
- Organization:
  - properties may include: name, org_type, surface_forms
- Place:
  - properties may include: name, place_type, surface_forms
- Document:
  - properties may include: title_or_label, doc_type, quoted_text, date_text, language
- Artifact:
  - properties may include: name, artifact_type, description
- Event:
  - properties may include: event_type, summary, date_text, sequence, certainty
- Case:
  - properties may include: case_name, summary, status
- Role:
  - properties may include: role_name, role_type, temporal_scope

Node requirements:
- Each node must have:
  - id: canonical human-readable identifier, preferably the most complete explicit form in the document
  - type: exactly one of the allowed node types above
  - properties: flat JSON object with primitive values only

Relationship requirements:
- Extract all important explicit relationships between nodes.
- Relationship types are restricted to the allowed list above.
- Each relationship must have:
  - source: node object
  - target: node object
  - type: exactly one of the allowed relationship types
  - properties: flat JSON object with primitive values only

Event extraction guidance:
- Create event nodes only when useful, e.g. a marriage, departure, attack, discovery, consultation, or theft.
- Link participants to events with relations such as PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE.
- Do not create trivial event nodes for every sentence.

Deduplication:
- Each real-world entity should appear only once per document after clear coreference resolution.
- Do not create duplicate nodes that differ only by shortened mentions when the fuller canonical mention is explicit and unambiguous in the same document.

Document object:
- Return the input document unchanged under the key "document".
- Preserve its metadata and page_content.

Output format:
Return exactly one JSON object with exactly these top-level keys:
- nodes
- relationships
- document
""",
        ),
        ("human", "Input document:\n{input_document}"),
    ]
)


# llm = ChatOllama(
#     model="qwen3.6:latest",
#     base_url="http://192.168.178.67:11434",
#     temperature=0,
#     # format=ExtractionResult.model_json_schema(),
#     reasoning=False,
#     keep_alive="24h",
#     num_predict=-1,
#     format="json"
# )

extraction_llm = ChatAnthropic(
    model_name="claude-sonnet-5",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

llm_json = extraction_llm.with_structured_output(ExtractionResult, include_raw=True)
json_chain = prompt | llm_json

# response = json_chain.invoke({"input_document":json.dumps(
#                 {
#                     "metadata": documents[2].metadata,
#                     "page_content": documents[2].page_content,
#                 },
#                 ensure_ascii=False,
#             )})

In [9]:
from copy import deepcopy
from typing import Dict, Tuple
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship

def _normalize_id(node_id: str, alias_map: Dict[str, str]) -> str:
    key = node_id.strip().lower()
    return alias_map.get(key, node_id.strip())

def _merge_properties(a: dict, b: dict) -> dict:
    merged = deepcopy(a) if a else {}
    for k, v in (b or {}).items():
        if k not in merged:
            merged[k] = v
        else:
            if merged[k] == v:
                continue
            if isinstance(merged[k], list):
                existing = merged[k]
            else:
                existing = [merged[k]]
            if isinstance(v, list):
                for item in v:
                    if item not in existing:
                        existing.append(item)
            else:
                if v not in existing:
                    existing.append(v)
            merged[k] = existing
    return merged

def _canonical_node(node: Node, alias_map: Dict[str, str]) -> Node:
    new_node = deepcopy(node)
    new_node.id = _normalize_id(str(node.id), alias_map)
    return new_node

def resolve_results(validated_result: GraphDocument) -> GraphDocument:
    alias_map = {
        "holmes": "Sherlock Holmes",
        "sherlock": "Sherlock Holmes",
        "mr. holmes": "Sherlock Holmes",
        "watson": "Dr. John H. Watson",
        "dr. watson": "Dr. John H. Watson",
        "the narrator": "Dr. John H. Watson",
        "narrator": "Dr. John H. Watson",
        "the woman": "Irene Adler",
        "miss adler": "Irene Adler",
        "irene norton": "Irene Adler",
        "i": "Dr. John H. Watson",
        "he": "Sherlock Holmes"
    }

    canonical_nodes: Dict[Tuple[str, str], Node] = {}

    for node in validated_result.nodes:
        cnode = _canonical_node(node, alias_map)
        key = (str(cnode.id), str(cnode.type))
        if key not in canonical_nodes:
            canonical_nodes[key] = cnode
        else:
            canonical_nodes[key].properties = _merge_properties(
                canonical_nodes[key].properties,
                cnode.properties
            )

    resolved_relationships = []

    for rel in validated_result.relationships:
        new_rel = deepcopy(rel)
        new_rel.source = _canonical_node(rel.source, alias_map)
        new_rel.target = _canonical_node(rel.target, alias_map)

        source_key = (str(new_rel.source.id), str(new_rel.source.type))
        target_key = (str(new_rel.target.id), str(new_rel.target.type))

        if source_key in canonical_nodes:
            new_rel.source = canonical_nodes[source_key]
        else:
            canonical_nodes[source_key] = new_rel.source

        if target_key in canonical_nodes:
            new_rel.target = canonical_nodes[target_key]
        else:
            canonical_nodes[target_key] = new_rel.target

        resolved_relationships.append(new_rel)

    deduped_relationships = []
    seen_rels = set()

    for rel in resolved_relationships:
        rel_key = (
            str(rel.source.id),
            str(rel.source.type),
            str(rel.type),
            str(rel.target.id),
            str(rel.target.type),
            json.dumps(rel.properties, sort_keys=True, ensure_ascii=False),
        )
        if rel_key not in seen_rels:
            seen_rels.add(rel_key)
            deduped_relationships.append(rel)

    return GraphDocument(
        nodes=list(canonical_nodes.values()),
        relationships=deduped_relationships,
        source=validated_result.source
    )

In [10]:
def extract_graph_document(resolved: ExtractionResult):
    return GraphDocument(nodes=resolved.nodes, relationships=resolved.relationships, source=resolved.document)

# def resolve_results(validated_result: GraphDocument):
#     resolved_nodes = []
#     resolved_relationships = []
#     alias_map = {
#         "holmes": "Sherlock Holmes",
#         "sherlock": "Sherlock Holmes",
#         "watson": "Dr. Watson",
#         "the woman": "Irene Adler",
#         "miss adler": "Irene Adler",
#         "mr. holmes": "Sherlock Holmes",
#         "i": "Dr. Watson",
#         "narrator": "Dr. Watson",
#         "he": "Sherlock Holmes"
#     }
#     for node in validated_result.nodes:
#         new_node = deepcopy(node)
#         if node.id.lower() in alias_map:
#             new_node.id = alias_map[node.id.lower()]
#             resolved_nodes.append(new_node)

#     for rel in validated_result.relationships:
#         source_node = rel.source
#         target_node = rel.target
#         new_rel = deepcopy(rel)
#         if source_node.id.lower() in alias_map:
#             new_node = deepcopy(source_node)
#             new_node.id = alias_map[source_node.id.lower()]
#             new_rel.source = new_node
#         if target_node.id.lower() in alias_map:
#             new_node = deepcopy(target_node)
#             new_node.id = alias_map[target_node.id.lower()]
#             new_rel.target = new_node
        
#         resolved_relationships.append(new_rel)
    
#     return GraphDocument(nodes=resolved_nodes, relationships=resolved_relationships, source=validated_result.source)


def extract_graph_document_json_enforced(doc: Document) -> GraphDocument:
    response = json_chain.invoke({"input_document": doc})
    # print(response)
    # json_data = json.loads(response.content)
    # print(f"json: {json_data}")

    validated = ExtractionResult.model_validate(response["parsed"])
    graph_doc = extract_graph_document(validated)
    return resolve_results(graph_doc)


def process_documents_json_enforced(documents: List[Document]) -> List[GraphDocument]:
    graph_docs = []
    for i, doc in enumerate(documents):
        try:
            graph_doc = extract_graph_document_json_enforced(doc)
            graph_docs.append(graph_doc)
            print(f"Processed document {i+1}/{len(documents)}")
        except (ValidationError, json.JSONDecodeError, Exception) as e:
            print(f"Failed document {i+1}: {e}")
    return graph_docs



In [74]:
# graph_docs = []
# for doc in documents[:5]:

# directory = "../data/chapters/"
# for file in os.listdir(directory):
#     print(f"processing {file}...")
loader = TextLoader(file_path=f"../data/chapters/chapter_1.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
documents = text_splitter.split_documents(documents=docs)

graph_docs = process_documents_json_enforced(documents)

with open(f"../data/gold_standard/gold_1_cloud.txt", "w") as file_out:
    file_out.write(str(graph_docs))

Processed document 1/53
Processed document 2/53
Processed document 3/53
Processed document 4/53
Processed document 5/53
Processed document 6/53
Processed document 7/53
Processed document 8/53
Processed document 9/53
Processed document 10/53
Processed document 11/53
Processed document 12/53
Processed document 13/53
Processed document 14/53
Processed document 15/53
Failed document 16: 1 validation error for ExtractionResult
  Input should be a valid dictionary or instance of ExtractionResult [type=model_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type
Processed document 17/53
Processed document 18/53
Processed document 19/53
Processed document 20/53
Processed document 21/53
Processed document 22/53
Processed document 23/53
Processed document 24/53
Processed document 25/53
Processed document 26/53
Processed document 27/53
Processed document 28/53
Processed document 29/53
Processed document 30/53
Processed document

In [25]:
graph.add_graph_documents(
    graph_docs,
    baseEntityLabel=True,
    include_source=True,
)

In [22]:
def clean_graph():
    query = """
    MATCH (n)
    DETACH DELETE n
    """
    graph.query(query)

In [ ]:
# import os
# import json
# from collections.abc import Mapping

# def flatten_dict(d, parent_key="", sep="_"):
#     items = {}
#     for k, v in d.items():
#         new_key = f"{parent_key}{sep}{k}" if parent_key else k
#         if isinstance(v, Mapping):
#             items.update(flatten_dict(v, new_key, sep=sep))
#         elif isinstance(v, list):
#             cleaned = []
#             for item in v:
#                 if isinstance(item, Mapping):
#                     cleaned.append(json.dumps(item, ensure_ascii=False))
#                 else:
#                     cleaned.append(item)
#             items[new_key] = cleaned
#         else:
#             items[new_key] = v
#     return items

# def sanitize_graph_document(graph_doc):
#     for node in graph_doc.nodes:
#         if node.properties:
#             node.properties = flatten_dict(node.properties)
#     for rel in graph_doc.relationships:
#         if rel.properties:
#             rel.properties = flatten_dict(rel.properties)
#     return graph_doc

# directory = "../data/cloud/"
# sanitized_docs = []
# for filename in os.listdir(directory):
#     with open(os.path.join(directory,filename), "r", encoding="utf-8") as file_in:
#         string_graph_docs = file_in.read()

parsed_graph_document = convert_to_graph_doc(graph_docs)
# for graph_doc in parsed_graph_document:
#     sanitized_docs.append(resolve_results(graph_doc))

# sanitized_docs.extend([sanitize_graph_document(doc) for doc in parsed_graph_document])

graph.add_graph_documents(
    parsed_graph_document,
    baseEntityLabel=True,
    include_source=True,
)
print(f"Added {filename} to graph.")

Added chapter_1.txt to graph.


In [69]:
len(parsed_graph_document)

53

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en",
    model_kwargs = {"device": "cpu"}
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    database="cloud4"
)
def invoke_vector_retriever(query:str, k:int=10):
    vector_retriever = vector_index.as_retriever(search_kwargs={"k": k})
    return vector_retriever.invoke(query)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

In [14]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` IF NOT EXISTS
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session(database="cloud4") as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    print("Index creation failed")
    pass

# Close the driver connection
driver.close()

Unable to retrieve routing information
Transaction failed and will be retried in 0.8587102496543252s (Unable to retrieve routing information)
Unable to retrieve routing information
Transaction failed and will be retried in 2.27779127456769s (Unable to retrieve routing information)
Unable to retrieve routing information
Transaction failed and will be retried in 3.9304930390986734s (Unable to retrieve routing information)
Unable to retrieve routing information
Transaction failed and will be retried in 8.863441748960492s (Unable to retrieve routing information)


Index creation failed


In [12]:
class Entities(BaseModel):
    """Entity mentions relevant for graph retrieval."""
    names: list[str] = Field(
        ...,
        description="All entity names explicitly mentioned in the question that may appear as graph nodes."
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You extract graph entity mentions from questions. Return all explicitly mentioned entities that may exist as graph nodes."
    ),
    (
        "human",
        "Extract entities from this question: {question}"
    ),
])


entity_chain = prompt | cloud_llm.with_structured_output(Entities, method="function_calling")

In [5]:
entity_chain.invoke("Who is Sherlock Holmes' assistant?")

Entities(names=['Sherlock Holmes'])

In [13]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


def graph_retriever(question: str, k: int = 10) -> str:
    result = []
    entities = entity_chain.invoke(question)
    if not entities:
        raise Exception("No entities present")

    for entity in entities.names:
        response = graph.query(
        """
            CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})
            YIELD node, score
            CALL {
                WITH node
                MATCH (node)-[r:!MENTIONS]->(neighbor)
                RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
                UNION
                WITH node
                MATCH (node)<-[r:!MENTIONS]-(neighbor)
                RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output
            LIMIT 50
        """,
            {"query": generate_full_text_query(entity)},
        )

        result.extend(row["output"] for row in response)

    return "\n".join(sorted(set(result))[:k])

In [35]:
graph_retriever("Irene Adler")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Irene~2 AND Adler~2


'0461b8e746498e4ccf3d4cd335c3353d - MENTIONS -> Irene Adler\n0fd1790d9a99ad28a593a3c10896f780 - MENTIONS -> Irene Adler\n3aec51217e7d835fdd61fbfc680fbabe - MENTIONS -> Irene Adler\n41dd52abf7eaf43b0bdfac6cca22204f - MENTIONS -> Irene Adler\n54042ec1404ed936346631655de1a344 - MENTIONS -> Irene Adler\n5ea0cffc90563ba4ec0c4e896dfb59b8 - MENTIONS -> Irene Adler\n739f30db4c7509289a28e192ee32a072 - MENTIONS -> Irene Adler\n7b6c3f8ccecd8666bd1bc08fb88522a1 - MENTIONS -> Irene Adler\n7f5496fdd807d5e3d96ee22817b64c22 - MENTIONS -> Irene Adler\n96548eaf3f4c71fac2e5449bfeace62f - MENTIONS -> Irene Adler'

In [12]:
def full_retriever(question: str) -> str:
    graph_data = graph_retriever(question, k=5)
    doc_results = invoke_vector_retriever(question, k=5)
    vector_data = [d.page_content for d in doc_results]

    return f"""Graph data:
{graph_data}

Document data:
{"#Document ".join(vector_data)}
"""

In [13]:
prompt_template = """Answer the question based only on the following context.

Context: {context}

Question: {query}

If a direct graph relation answers the question, prefer it.
Use natural language and be as precise and concise as possible.
Answer as short as possible without omitting any relevant information.
Leave out any explanation and reasoning.

Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

In [94]:
prompt_template = """You are a precise assistant answering questions strictly from the given context, which may include direct graph relations (subject-predicate-object triples) and text passages.

Context:
{context}

Question:
{query}

Instructions:
- If a graph relation directly answers the question, state it as a short, natural-language sentence (not a bare entity).
- Answer in one concise sentence that restates the key subject of the question.
- Use only facts explicitly present in the context; do not infer or add outside knowledge.
- If the context does not contain enough information, respond: "The context does not provide this information."
- Do not include reasoning, explanations, or meta-commentary.

Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
prompt_template = """
You are an assistant that answers questions using only the provided context.

Rules:
- Answer with the shortest possible correct answer.
- If a direct graph relation answers the question, prefer it.
- Output only the answer itself.
- Do not explain your answer.
- Do not repeat the question.
- Do not add background, reasoning, or extra words.
- Do not quote the context unless the answer is a direct quote.
- Use only information explicitly stated in the context.
- If the answer is not supported by the context, respond exactly with: I cannot answer based on the context.

Context:
{context}

Question:
{query}

Answer:
"""


In [103]:
print(prompt_template)

Answer the question based only on the following context.

Context: {context}

Question: {query}

If a direct graph relation answers the question, prefer it.
Use natural language and be as precise and concise as possible.
Answer as short as possible without omitting any relevant information.
Leave out any explanation and reasoning.

Answer:


In [14]:
prompt = ChatPromptTemplate.from_template(prompt_template)

chain = (
        {
            "context": full_retriever,
            "query": RunnablePassthrough(),
        }
    | prompt
    | cloud_llm
    | StrOutputParser()
)

In [39]:
chain.invoke(input="Where does Holmes believe the compromising photograph is hidden after watching Briony Lodge?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Holmes~2
Generated Query: Briony~2 AND Lodge~2
Generated Query: compromising~2 AND photograph~2


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

"The context does not provide information about where Holmes believes the compromising photograph is hidden after watching Briony Lodge. The documents describe Holmes's reconnaissance of the house and the subsequent events, but they do not explicitly state his conclusion about the photograph's location."

In [15]:
def format_docs(relevant_docs):
    # return "\n".join(doc.payload["page_content"] for doc in relevant_docs)
    return "\n".join(doc.page_content for doc in relevant_docs)


def answer_with_context(query):
    context = full_retriever(query)

    # docs =  retrieve_with_reranking(query, k=20)
    # context = format_docs(docs)

    answer = (prompt | cloud_llm | StrOutputParser()).invoke(
        {"context": context, "query": query}
    )

    return {
        "query": query,
        "context": context,
        "answer": answer,
        "docs": docs,
    }

In [16]:
import json

# qa_path = "../data/qa/"
# for file in os.listdir(qa_path):
with open(f"../data/qa/qa_chapter_1_pairs_sonnet_5.json") as f:
    json_qa = json.loads(f.read())



In [43]:
len(json_qa)

20

In [30]:
hybrid_qa_data = []
for pair in json_qa:
    vector_context = [el.page_content for el in invoke_vector_retriever(pair["question"], k=5)]
    graph_text = graph_retriever(pair["question"], k=10)
    graph_context = [line for line in graph_text.split("\n") if line.strip()]
    hybrid_context = graph_context + vector_context

    answer = (prompt | cloud_llm | StrOutputParser()).invoke(
        {"context": hybrid_context, "query": pair["question"]}
    )

    hybrid_qa_data.append(
        {
            "user_input": pair["question"],
            # "retrieved_contexts": [res.payload["page_content"] for res in response["docs"]],
            "retrieved_contexts": hybrid_context,
            "response": answer,
            "reference": pair["answer"],
        }
    )
    print(f"processed {len(hybrid_qa_data)}/{len(json_qa)} qa pairs")
hybrid_evaluation_dataset = EvaluationDataset.from_list(hybrid_qa_data)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Sherlock~2 AND Holmes~2
Generated Query: the~2 AND woman~2
processed 1/20 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Holmes~2
Generated Query: Holland~2
processed 2/20 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

...

Generated Query: Holmes~2
Generated Query: Irene~2 AND Adler~2
processed 18/20 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Holmes~2
Generated Query: King~2
processed 19/20 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Irene~2 AND Adler~2
Generated Query: Holmes~2
processed 20/20 qa pairs


In [31]:
for q in hybrid_qa_data:
    print("-------------------------")
    print(f"Question: {q["user_input"]}")
    print(f"Response: {q["response"]}")
    print(f"Reference: {q["reference"]}")

-------------------------
Question: Who does the narrator say is always referred to by Sherlock Holmes as 'the woman'?
Response: Irene Adler is always referred to by Sherlock Holmes as 'the woman'.
Reference: Irene Adler is the woman Sherlock Holmes always refers to as 'the woman'; she eclipses the whole of her sex in his eyes.
-------------------------
Question: What two past cases of Holmes are mentioned besides the mission for the royal family of Holland?
Response: The two past cases mentioned besides the Holland royal family mission are:

1. The Trepoff murder in Odessa
2. The Atkinson brothers tragedy at Trincomalee
Reference: Holmes's summons to Odessa for the Trepoff murder and his clearing up of the tragedy of the Atkinson brothers at Trincomalee.
-------------------------
Question: On what date did Watson pass by Baker Street and feel a desire to see Holmes again?
Response: March 20, 1888
Reference: It was on the twentieth of March, 1888.
-------------------------
Question: Ho

1: correct
2: correct
3: correct
4: missed key fact, otherwise correct context
5: correct
6: correct, long
7: correct, long
8: incorrect
9: correct
10: incorrect
11: incorrect
12: correct
13: incorrect
14: incorrect
15: incorrect
16: incorrect
17: correct
18: incorrect
19: correct
20: correct, long

In [32]:
from ragas.metrics import (
    context_precision,
    context_recall,
    context_entity_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness
)
from ragas.metrics import NoiseSensitivity

evaluation_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
judge_llm = LangchainLLMWrapper(evaluation_llm)
judge_embedding = LangchainEmbeddingsWrapper(embeddings)

noise_sensitivity_relevant = NoiseSensitivity(mode="relevant")

scores = evaluate(
    hybrid_evaluation_dataset,
    metrics=[
        # answer_correctness,
        # answer_relevancy,
        # faithfulness,
        # context_precision,
        context_recall,
        # context_entity_recall,
        # noise_sensitivity_relevant,
    ],
    llm=judge_llm,
    embeddings=judge_embedding,
)
print(scores)

/tmp/ipykernel_763710/1768582928.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_763710/1768582928.py:1: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_763710/1768582928.py:1: DeprecationWarning: Importing context_entity_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_entity_recall
  from ragas.metrics import (
/tmp/ipykernel_763710/1768582928.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

{'context_recall': 0.6550}


In [ ]:
from rapidfuzz import fuzz, process
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship


def normalize(s: str) -> str:
    return s.strip().lower()


def fuzzy_match_nodes(
    gold_graph: GraphDocument,
    pred_graph: GraphDocument,
    threshold: float = 90.0,
) -> dict:
    # Prepare normalized lists per type
    gold_nodes = [(normalize(n.id), n.type, n) for n in gold_graph.nodes]
    pred_nodes = [(normalize(n.id), n.type, n) for n in pred_graph.nodes]

    # Group by type to avoid matching person to location, etc.
    gold_by_type = {}
    for norm_id, ntype, node in gold_nodes:
        gold_by_type.setdefault(ntype, []).append((norm_id, node))

    pred_by_type = {}
    for norm_id, ntype, node in pred_nodes:
        pred_by_type.setdefault(ntype, []).append((norm_id, node))

    tp = 0
    matched_gold = set()
    matched_pred = set()

    # For each type, fuzzy match predicted ids to gold ids
    for ntype, pred_list in pred_by_type.items():
        if ntype not in gold_by_type:
            continue
        gold_list = gold_by_type[ntype]

        gold_ids = [g_id for g_id, _ in gold_list]

        for pred_idx, (pred_id, pred_node) in enumerate(pred_list):
            # Use RapidFuzz to find best match among gold ids
            match = process.extractOne(
                pred_id,
                gold_ids,
                scorer=fuzz.token_sort_ratio,  # good for multi-word strings
            )
            if match is None:
                continue

            best_gold_id, score, gold_pos = match
            if score >= threshold:
                # Record the match
                tp += 1
                matched_pred.add((ntype, pred_idx))
                matched_gold.add((ntype, gold_pos))

            # else:
            #     print(f"{pred_id}")

    gold_count = len(gold_nodes)
    pred_count = len(pred_nodes)

    fn = gold_count - tp
    fp = pred_count - tp

    # for gold_node in gold_nodes:
    #     relevant = gold_by_type.get(gold_node[1])
    #     if gold_node[0] not in relevant:
    #         print(gold_node)

    precision = tp / pred_count if pred_count > 0 else 0.0
    recall = tp / gold_count if gold_count > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold_node_count": gold_count,
        "pred_node_count": pred_count,
        "tp_nodes": tp,
        "fp_nodes": fp,
        "fn_nodes": fn,
    }


# def fuzzy_match_relationships(
#     gold_graph: GraphDocument, pred_graph: GraphDocument, threshold: int = 60
# ) -> dict:
#     gold_by_type: Dict[str, list[Tuple[str, str]]] = {}
#     tp = 0

#     # Prepare gold by type
#     for r in gold_graph.relationships:
#         rel_type = r.type.strip()
#         src_id = normalize(r.source.id)
#         tgt_id = normalize(r.target.id)
#         gold_by_type.setdefault(rel_type, []).append((src_id, tgt_id))

#     gold_rel_count = len(gold_graph.relationships)
#     pred_rel_count = len(pred_graph.relationships)

#     # Fuzzy matching
#     for pred_rel in pred_graph.relationships:
#         rel_type = pred_rel.type.strip()
#         gold_match_type = gold_by_type.get(rel_type)
#         if gold_match_type is None:
#             continue

#         pred_src = normalize(pred_rel.source.id)
#         pred_tgt = normalize(pred_rel.target.id)

#         for gold_src, gold_tgt in gold_match_type:
#             if (
#                 fuzz.ratio(pred_src, gold_src) >= threshold
#                 and fuzz.ratio(pred_tgt, gold_tgt) >= threshold
#             ) or (
#                 fuzz.ratio(pred_src, gold_tgt) >= threshold
#                 and fuzz.ratio(pred_tgt, gold_src) >= threshold # also check for inverted rel
#             ):
#                 tp += 1
#                 break  # count at most one match per predicted relation
            
#     fp = pred_rel_count - tp
#     fn = gold_rel_count - tp

#     precision = tp / pred_rel_count if pred_rel_count > 0 else 0.0
#     recall = tp / gold_rel_count if gold_rel_count > 0 else 0.0
#     f1 = (
#         2 * precision * recall / (precision + recall)
#         if (precision + recall) > 0
#         else 0.0
#     )

#     return {
#         "precision": precision,
#         "recall": recall,
#         "f1": f1,
#         "gold_rel_count": gold_rel_count,
#         "pred_rel_count": pred_rel_count,
#         "tp_rels": tp,
#         "fp_rels": fp,
#         "fn_rels": fn,
#     }

def fuzzy_match_relationships(
    gold_graph: GraphDocument, pred_graph: GraphDocument, threshold: int = 60
) -> dict:
    gold_by_type: Dict[str, list[Tuple[str, str]]] = {}
    tp = 0

    for r in gold_graph.relationships:
        rel_type = r.type.strip()
        src_id = normalize(r.source.id)
        tgt_id = normalize(r.target.id)
        gold_by_type.setdefault(rel_type, []).append((src_id, tgt_id))

    # Track which gold relations (by type + index) have been matched
    matched_gold_keys = set()
    unmatched_pred = []

    gold_rel_count = len(gold_graph.relationships)
    pred_rel_count = len(pred_graph.relationships)

    for pred_rel in pred_graph.relationships:
        rel_type = pred_rel.type.strip()
        gold_match_type = gold_by_type.get(rel_type)

        if gold_match_type is None:
            unmatched_pred.append(pred_rel)
            continue

        pred_src = normalize(pred_rel.source.id)
        pred_tgt = normalize(pred_rel.target.id)

        matched = False
        for idx, (gold_src, gold_tgt) in enumerate(gold_match_type):
            if (
                fuzz.ratio(pred_src, gold_src) >= threshold
                and fuzz.ratio(pred_tgt, gold_tgt) >= threshold
            ) or (
                fuzz.ratio(pred_src, gold_tgt) >= threshold
                and fuzz.ratio(pred_tgt, gold_src) >= threshold
            ):
                tp += 1
                matched_gold_keys.add((rel_type, idx))
                matched = True
                break

        if not matched:
            unmatched_pred.append(pred_rel)

    unmatched_pred.sort(key=lambda x: x.type, reverse=True)

    for rel in unmatched_pred:
        print(f"{rel.type, rel.source.id, rel.target.id}")

    # Determine unmatched gold relations
    unmatched_gold = []
    for rel_type, rels in gold_by_type.items():
        for idx, (src, tgt) in enumerate(rels):
            if (rel_type, idx) not in matched_gold_keys:
                unmatched_gold.append((rel_type, src, tgt))

    print("#######################################################")
    for rel in unmatched_gold:
            print(f"{rel[0], rel[1], rel[2]}")

    fp = pred_rel_count - tp
    fn = gold_rel_count - tp

    precision = tp / pred_rel_count if pred_rel_count > 0 else 0.0
    recall = tp / gold_rel_count if gold_rel_count > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold_rel_count": gold_rel_count,
        "pred_rel_count": pred_rel_count,
        "tp_rels": tp,
        "fp_rels": fp,
        "fn_rels": fn,
        "unmatched_pred": len(unmatched_pred),
        "unmatched_gold": len(unmatched_gold),
    }


In [45]:
def merge_graph_documents(graph_docs: list[GraphDocument]) -> GraphDocument:
    resolved_graph_docs = []
    for gd in graph_docs:
        resolved_graph_docs.append(resolve_results(gd))

    all_nodes = []
    all_rels = []
    # You can pick any source; here we just keep the first document's source
    source = graph_docs[0].source if graph_docs else None

    for gd in graph_docs:
        all_nodes.extend(gd.nodes)
        all_rels.extend(gd.relationships)

    return GraphDocument(nodes=all_nodes, relationships=all_rels, source=source)

def get_nodes_ids(graph_docs:any):
    # resolve graph docs
    resolved_graph_docs = []
    for gd in graph_docs:
        resolved_graph_docs.append(resolve_results(gd))
    # merge graph docs
    merged_graph_docs = merge_graph_documents(resolved_graph_docs)

    # return only node ids
    all_nodes_names = []
    for node in merged_graph_docs.nodes:
        if node.id not in all_nodes_names:
            all_nodes_names.append(node.id)
    return all_nodes_names

def get_rel_types(graph_docs:any):
    all_rel_types = []
    for rel in merge_graph_documents(graph_docs).relationships:
        if rel.type not in all_rel_types:
            all_rel_types.append(rel.type)
    return all_rel_types

In [46]:
def evaluate_graph_quality(gold_graph: GraphDocument, pred_graph: GraphDocument) -> dict:
    node_scores = fuzzy_match_nodes(gold_graph, pred_graph, 60)
    rel_scores = fuzzy_match_relationships(gold_graph, pred_graph)
    return {
        "nodes": node_scores,
        "relationships": rel_scores,
    }

In [18]:
directory = "../data/cloud/"
with open(os.path.join(directory, "chapter_1_cloud.txt"), "r", encoding="utf-8") as file_in:
    string_graph_docs = file_in.read()

# with open(os.path.join(directory,"chapter_1.txt"), "r", encoding="utf-8") as file_in:

parsed_graph_document = convert_to_graph_doc(string_graph_docs)

In [51]:
extracted_docs = parsed_graph_document

with open("../data/gold_standard/gold_1_cloud.txt", "r", encoding="utf-8") as file_in:
    gold_standard_text = file_in.read()

gold_standard_docs = convert_to_graph_doc(gold_standard_text)

pred_graph_ch1 = merge_graph_documents(extracted_docs)
extracted_ids = get_nodes_ids(extracted_docs)
print(extracted_ids)
extracted_types = get_rel_types(extracted_docs)
print(extracted_types)


gold_graph_ch1 = merge_graph_documents(gold_standard_docs)
gold = get_nodes_ids(gold_standard_docs)
gold_types = get_rel_types(gold_standard_docs)
scores_ch1 = evaluate_graph_quality(gold_graph_ch1, pred_graph_ch1)
print(json.dumps(scores_ch1, indent=2, ensure_ascii=False))

['Sherlock Holmes', 'Irene Adler', 'Dr. John H. Watson', 'Baker Street', 'Odessa', 'Trincomalee', 'Holland', 'Trepoff murder', 'Atkinson brothers tragedy', 'Holland royal family mission', 'Atkinson brothers', 'reigning family of Holland', 'the reigning family of Holland', 'the daily press', "the narrator's former friend and companion", "Holmes's rooms", 'Study in Scarlet', 'March 20, 1888', 'armchair', 'case of cigars', 'spirit case', 'gasogene', 'servant girl', 'Mary Jane', "Watson's wife", 'the speaker', 'the gentleman visitor', 'the London slavey', "the speaker's rooms", 'left shoe', 'top-hat', 'stethoscope', 'the mysterious gentleman', 'royal houses of Europe', 'the note', 'the seventeen steps', 'consultation event', 'the paper', 'the unknown writer', 'Wallenstein', 'Egria', 'Bohemia', 'Carlsbad', 'Continental Gazetteer', 'paper_specimen', 'Unknown German', 'Note', 'Brougham', 'Case', 'the client', 'stairs', 'passage', 'door', 'The Tall Visitor', 'Black Vizard Mask', 'Brooch', 'Dou

In [82]:
unknown_rel_type = []
print(gold_types)
for rel in pred_graph_ch1.relationships:
    if rel.type not in gold_types:
        unknown_rel_type.append(rel.type)
print(len(unknown_rel_type))
print(unknown_rel_type)

['RELATED_TO_CASE', 'AFFILIATED_WITH', 'RESIDES_AT', 'OCCURRED_AT', 'PARTICIPATED_IN', 'TARGETS', 'LOCATED_IN', 'POSSESSES', 'OCCURRED_ON', 'WORKS_FOR', 'ADDRESSED_TO', 'CREATED', 'ALIAS_OF', 'HAS_ROLE']
0
[]


In [36]:
pred_by_type: Dict[str, list[Tuple[str, str]]] = {}

# Prepare gold by type
for r in gold_graph_ch1.relationships:
    rel_type = r.type.strip()
    src_id = normalize(r.source.id)
    tgt_id = normalize(r.target.id)
    pred_by_type.setdefault(rel_type, []).append((src_id, tgt_id))

for t in pred_by_type:
    print(t)
    ids = pred_by_type.get(t)
    for i in ids:
        print(i)
    print("------------------------------")


RELATED_TO_CASE
('sherlock holmes', 'irene adler')
('sherlock holmes', 'trepoff murder case')
('sherlock holmes', 'atkinson brothers tragedy')
('atkinson brothers', 'atkinson brothers tragedy')
('sherlock holmes', 'holland mission')
('holland mission', 'reigning family of holland')
('dr. john h. watson', 'study in scarlet')
('sherlock holmes', 'new problem')
('sherlock holmes', 'dr. john h. watson')
('top-hat', 'stethoscope')
('sherlock holmes', 'left shoe')
('sherlock holmes', 'dr. john h. watson')
('anonymous note', 'delivery of the note')
('sherlock holmes', 'note from client')
('consultation with holmes', 'irene adler')
('the letters', 'the king')
('the photograph', 'the king')
("case of the king's photograph", 'the king')
("case of the king's photograph", 'the young person')
("case of the king's photograph", 'sherlock holmes')
("case of the king's photograph", 'the photograph')
("case of the king's photograph", 'the letters')
('clotilde lothman von saxe-meningen', 'king of scandin